# DoE Lite

Alexander Dowling (adowling@nd.edu, 2025)

The goal of this notebook is to prototype a "Pyomo.DoE lite" that uses symbolic differentiation instead of finite difference to assemble the sensitivity matrix.

## Define Simple Experiment Model

We will reuse the reactor experiment example.

In [1]:
from scipy.interpolate import interp1d

import pyomo.environ as pyo
import idaes # Load solvers

from pyomo.contrib.doe.examples.reactor_experiment import ReactorExperiment # Load the example

# import json

# Copied from the json file in the example
# TODO: Use the json file instead of copying the data. Need to figure out how to load the file without hardcoding the path.
data_ex = {"CA0": 5.0, "CA_bounds": [1.0, 5.0], "CB0": 0.0, "CC0": 0.0, "t_range": [0.0, 1.0], "control_points": {0: 500, 0.125: 300, 0.25: 300, 0.375: 300, 0.5: 300, 0.625: 300, 0.75: 300, 0.875: 300, 1: 300}, "T_bounds": [300, 700], "A1": 84.79, "A2": 371.72, "E1": 7.78, "E2": 15.05}

experiment = ReactorExperiment(data=data_ex, nfe=10, ncp=3)

model = experiment.get_labeled_model()

# Fix the experiment inputs
# TODO: This can get generalized

control_points = data_ex["control_points"]

for t in model.t:
    if t in control_points.keys():
        val = control_points[t]
        model.T[t].fix(val)
    else:
        continue

# Fix the initial conditions
model.CA[0].fix(data_ex["CA0"])

solver = pyo.SolverFactory('ipopt')

results = solver.solve(model, tee=True)

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

Confirms that we have a square model.

In [2]:
model.pprint()

1 Param Declarations
    R : Size=1, Index=None, Domain=Any, Default=None, Mutable=False
        Key  : Value
        None : 8.314

10 Var Declarations
    A1 : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 : 84.79 :  None :  True :  True : NonNegativeReals
    A2 : Size=1, Index=None
        Key  : Lower : Value  : Upper : Fixed : Stale : Domain
        None :     0 : 371.72 :  None :  True :  True : NonNegativeReals
    CA : Size=31, Index=t
        Key      : Lower : Value               : Upper : Fixed : Stale : Domain
               0 :   1.0 :                 5.0 :   5.0 :  True :  True : NonNegativeReals
        0.009691 :     0 :    4.40754994378421 :  None : False : False : NonNegativeReals
        0.040309 :     0 :   2.953717779095515 :  None : False : False : NonNegativeReals
          0.0625 :     0 :  2.2121699600378024 :  None : False : False : NonNegativeReals
        0.072191 :     0 :  1.9604030470289575 :  None :

## Assemble the Jacobians via Symbolic Difference

In [3]:
from pyomo.core.expr.calculus.diff_with_pyomo import reverse_sd
from pyomo.core.expr.visitor import identify_variables
from pyomo.common.collections import ComponentSet

# Get sets from the annocated model
param_set = ComponentSet()

for p in model.unknown_parameters.keys():
    param_set.add(p)

param_list = list(param_set)

print(param_list)


[<pyomo.core.base.var.ScalarVar object at 0x16354fbc0>, <pyomo.core.base.var.ScalarVar object at 0x1635c80b0>, <pyomo.core.base.var.ScalarVar object at 0x1635c8040>, <pyomo.core.base.var.ScalarVar object at 0x1635c8120>]


In [4]:
output_set = ComponentSet()
for o in model.experiment_outputs.keys():
    output_set.add(o)

output_list = list(output_set)
print(output_list)

[<pyomo.core.base.var.VarData object at 0x16354dd20>, <pyomo.core.base.var.VarData object at 0x1635c8510>, <pyomo.core.base.var.VarData object at 0x1635c86d0>, <pyomo.core.base.var.VarData object at 0x1635c87b0>, <pyomo.core.base.var.VarData object at 0x1635c8820>, <pyomo.core.base.var.VarData object at 0x1635c8890>, <pyomo.core.base.var.VarData object at 0x1635c8900>, <pyomo.core.base.var.VarData object at 0x1635c8970>, <pyomo.core.base.var.VarData object at 0x16354fca0>, <pyomo.core.base.var.VarData object at 0x16354fd10>, <pyomo.core.base.var.VarData object at 0x1635c91c0>, <pyomo.core.base.var.VarData object at 0x1635c9380>, <pyomo.core.base.var.VarData object at 0x1635c9460>, <pyomo.core.base.var.VarData object at 0x1635c94d0>, <pyomo.core.base.var.VarData object at 0x1635c9540>, <pyomo.core.base.var.VarData object at 0x1635c95b0>, <pyomo.core.base.var.VarData object at 0x1635c9620>, <pyomo.core.base.var.VarData object at 0x16354fd80>, <pyomo.core.base.var.VarData object at 0x1635

In [5]:
# first sort the variables
con_set = ComponentSet()
var_set = ComponentSet()
for c in model.component_data_objects(pyo.Constraint, descend_into=True, active=True):
    con_set.add(c)
    for v in identify_variables(c.body, include_fixed=False):
        var_set.add(v)
        
# recall that the parameters are fixed, so we did not
# get them above. Let's add them now.
for p in model.unknown_parameters.keys():
    var_set.add(p)

con_list = list(con_set)
var_list = list(var_set)

In [6]:
# get the Jacobian
jac_dict = {}
for i,c in enumerate(con_list):
    assert c.equality
    der_map = reverse_sd(c.body)
    for j,v in enumerate(var_list):
        if v in der_map:
            deriv = der_map[v]
        else:
            deriv = 0
        jac_dict[(i, j)] = deriv


## Build Constraints

In [7]:
# Grab all of the variables that are not parameters

param_index = []
model_var_index = []
measurement_index = []

# Loop over the variables
# and figure out which ones (and associated indices) are parameters 
# or mesaurements
for i, v in enumerate(var_set):
    if v in param_set:
        param_index.append(i)
    else:
        model_var_index.append(i)
        if v in output_set:
            measurement_index.append(i)

model.param_index = pyo.Set(initialize=param_index)
model.measurement_index = pyo.Set(initialize=measurement_index)
model.constraint_index = pyo.Set(initialize=range(len(con_list)))
model.var_index = pyo.Set(initialize=model_var_index)



In [8]:
model.param_index.pprint()

param_index : Size=1, Index=None, Ordered=Insertion
    Key  : Dimen : Domain : Size : Members
    None :     1 :    Any :    4 : {175, 176, 177, 178}


In [9]:
model.constraint_index.pprint()

constraint_index : Size=1, Index=None, Ordered=Insertion
    Key  : Dimen : Domain : Size : Members
    None :     1 :    Any :  175 : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174}


In [10]:
model.var_index.pprint()

var_index : Size=1, Index=None, Ordered=Insertion
    Key  : Dimen : Domain : Size : Members
    None :     1 :    Any :  175 : {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174}


In [11]:
model.jac_variables_wrt_param = pyo.Var(model.var_index, model.param_index, initialize=0)

# This has an index mistake... jac_dict includes the parameters, but var_index skips them
# We need to be more careful about the indices
@model.Constraint(model.constraint_index, model.param_index)
def jacobian_constraint(model, i, j):
    return jac_dict[(i,j)] == -sum(model.jac_variables_wrt_param[k,j] * jac_dict[(i,k)] for k in model.var_index)

In [12]:
results2 = solver.solve(model, tee=True)

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

In [13]:
for y in model.measurement_index:
    for p in model.param_index:
        print(f"Jacobian of {var_list[y]} with respect to {var_list[p]}: {model.jac_variables_wrt_param[y,p].value}")

Jacobian of CA[0.125] with respect to A1: -0.019279275229875433
Jacobian of CA[0.125] with respect to A2: -3.264524572832144e-38
Jacobian of CA[0.125] with respect to E1: 0.397875572600254
Jacobian of CA[0.125] with respect to E2: -1.223020271346915e-36
Jacobian of CA[0.25] with respect to A1: -0.015686103589098287
Jacobian of CA[0.25] with respect to A2: -2.039930055747123e-38
Jacobian of CA[0.25] with respect to E1: 0.37203304672491294
Jacobian of CA[0.25] with respect to E2: -1.1813339318738483e-36
Jacobian of CA[0.375] with respect to A1: -0.012084060916639291
Jacobian of CA[0.375] with respect to A2: -1.2763997388038562e-38
Jacobian of CA[0.375] with respect to E1: 0.30987226468791074
Jacobian of CA[0.375] with respect to E2: -6.8861050643136655e-37
Jacobian of CA[0.5] with respect to A1: -0.008982344704982418
Jacobian of CA[0.5] with respect to A2: -8.005031967703153e-39
Jacobian of CA[0.5] with respect to E1: 0.24217292342850247
Jacobian of CA[0.5] with respect to E2: -3.0724152

In [14]:
model.fim = pyo.Var(model.param_index, model.param_index, initialize=1)

@model.Constraint(model.param_index, model.param_index)
def fim_constraint(model, i, j):
    return model.fim[i,j] == sum(model.jac_variables_wrt_param[k,i] * model.jac_variables_wrt_param[k,j] for k in model.var_index)

results3 = solver.solve(model, tee=True)

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

## A-Optimality

In [15]:
@model.Objective(sense=pyo.maximize)
def trace_fim(model):
    return sum(model.fim[i,i] for i in model.param_index)

# unfix the control decisions
for t in model.t:
    if t in control_points.keys():
        val = control_points[t]
        model.T[t].unfix()
    else:
        continue

# unfix the initial conditions
model.CA[0].unfix()

results4 = solver.solve(model, tee=True)

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

## D-Optimality

In [16]:
# Copied from Pyomo.DoE

import numpy as np

# Toggle off A-opt objective
model.trace_fim.deactivate()

# Extract the FIM matrix
fim = np.zeros((len(model.param_index), len(model.param_index)))
for i, c in enumerate(model.param_index):
    for j, d in enumerate(model.param_index):
        fim[i, j] = model.fim[c, d].value
# Convert the FIM matrix to a numpy array
fim = np.array(fim)

# Calculate the eigenvalues of the FIM matrix
eig = np.linalg.eigvals(fim)

# If the smallest eigenvalue is (practically) negative, add a diagonal matrix to make it positive definite
small_number = 1e-10
if min(eig) < small_number:
    fim = fim + np.eye(len(model.param_index)) * (
        small_number - min(eig)
    )

# Compute the Cholesky decomposition of the FIM matrix
L = np.linalg.cholesky(fim)

model.L = pyo.Var(
                model.param_index, model.param_index, initialize=0
            )

# loop over parameter name
for i, c in enumerate(model.param_index):
    for j, d in enumerate(model.param_index):
        # fix the 0 half of L matrix to be 0.0
        if i < j:
            model.L[c, d].fix(0.0)
        # Give LB to the diagonal entries
        elif i == j:
            # Set the lower bound for the diagonal entries
            # to be a small number
            model.L[c, d].setlb(1E-10)
                    

# Initialize the Cholesky matrix
for i, c in enumerate(model.param_index):
    for j, d in enumerate(model.param_index):
        model.L[c, d].value = L[i, j]

def cholesky_imp(m, c, d):
    """
    Calculate Cholesky L matrix using algebraic constraints
    """
    # If the row is greater than or equal to the column, we are in the
    # lower triangle region of the L and FIM matrices.
    # This region is where our equations are well-defined.
    if list(m.param_index).index(c) >= list(m.param_index).index(d):
        return m.fim[c, d] == sum(
            m.L[c, m.param_index.at(k + 1)]
            * m.L[d, m.param_index.at(k + 1)]
            for k in range(list(m.param_index).index(d) + 1)
        )
    else:
        # This is the empty half of L above the diagonal
        return pyo.Constraint.Skip

model.cholesky_cons = pyo.Constraint(
    model.param_index, model.param_index, rule=cholesky_imp
)

model.logdet_FIM = pyo.Objective(
    expr=2 * sum(pyo.log10(model.L[j, j]) for j in model.param_index),
    sense=pyo.maximize,
)

In [17]:
results5 = solver.solve(model, tee=True)

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://